IMPORT CITY_DF FROM SQL

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# 1. Set connection
USER = 'root'
PASSWORD = 'insertyourpassword'
HOST = '127.0.0.1'
DATABASE = 'gans_database'

# 2. Create the engine
engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}/{DATABASE}")

# 3. Read city_df
df = pd.read_sql("SELECT * FROM city", con=engine)

# 4. Display df
print(df.head())


   city_id city_name  country  latitude  longitude
0        1    Berlin  Germany   52.5200     13.405
1        2   Hamburg  Germany   53.5500     10.000
2        3    Munich  Germany   48.1375     11.575


FIND ALL AIRPORTS FOR EACH LOCATION

In [ ]:
import requests
import time

url0 = "https://aerodatabox.p.rapidapi.com/airports/search/location"

headers0 = {
    'x-rapidapi-host': 'aerodatabox.p.rapidapi.com',
    'x-rapidapi-key': 'insertyourkeyhere'
}

responses = []

# For loop for each row
for index, row in df.iterrows():
    
    querystring = {
        "lat": str(row['latitude']), 
        "lon": str(row['longitude']), 
        "radiusKm": "50", 
        "limit": "10", 
        "withFlightInfoOnly": "false"
    }

    response = requests.get(url0, headers=headers0, params=querystring)
    #
    response_json = response.json()
    responses.append(response_json)
    #
    print(response_json)


    time.sleep(1.5)

--- Aeroporti vicino a Berlin ---
{'searchBy': {'lat': 52.52, 'lon': 13.405}, 'count': 3, 'items': [{'icao': 'EDDT', 'iata': 'TXL', 'name': 'Berlin -Tegel', 'shortName': '-Tegel', 'municipalityName': 'Berlin', 'location': {'lat': 52.5597, 'lon': 13.287699}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}, {'icao': 'EDDB', 'iata': 'BER', 'name': 'Berlin Brandenburg', 'shortName': 'Brandenburg', 'municipalityName': 'Berlin', 'location': {'lat': 52.35139, 'lon': 13.493889}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}, {'icao': 'EDAZ', 'iata': 'QXH', 'name': 'Trebbin Schönhagen', 'shortName': 'Schönhagen', 'municipalityName': 'Trebbin', 'location': {'lat': 52.20361, 'lon': 13.156389}, 'countryCode': 'DE', 'timeZone': 'Europe/Berlin'}]}
--- Aeroporti vicino a Hamburg ---
{'searchBy': {'lat': 53.55, 'lon': 10.0}, 'count': 2, 'items': [{'icao': 'EDDH', 'iata': 'HAM', 'name': 'Hamburg', 'shortName': 'Hamburg', 'municipalityName': 'Hamburg', 'location': {'lat': 53.6304, 'lon': 9.988229}

CREATE AIRPORTS_DF 

In [4]:
import pandas as pd

# 1. Create an empty list to gather all airport rows
airport_list = []

# Use zip to loop through both the API responses and your original DataFrame rows
for city_data, (index, row) in zip(responses, df.iterrows()):

    # Extract the items list containing all airports for THAT specific city
    airport_items = city_data.get("items", [])

    # THE SECOND LOOP: Iterate through each airport found in the items list
    for airport in airport_items:

        # Extract the 3-letter IATA code for the current airport
        iata_code = airport.get("iata")

        # Safety filter: only add to the DataFrame if a valid IATA code exists
        if iata_code:

            # Keep the exact original city name from your DataFrame (e.g., Berlin)
            row_data = {"city_name": row["city_name"], "arrival_iata": iata_code}

            # APPEND: Push the row into the global list
            airport_list.append(row_data)

# 2. Convert the full list of dictionaries into the final Pandas DataFrame
df_airports_gans = pd.DataFrame(airport_list)

# Display the clean vertical table
df_airports_gans


,city_name,arrival_iata
0,Berlin,TXL
1,Berlin,BER
2,Berlin,QXH
3,Hamburg,HAM
4,Hamburg,XFW
5,Munich,OBF
6,Munich,FEL
7,Munich,MUC


TRANSFER AIRPORTS TABLE ON SQL

In [ ]:
from sqlalchemy import text
import pandas as pd

# 1. Drop duplicates
df_airports_unique = df_airports_gans.drop_duplicates().copy()

# 2. Select 'city_id' and "city_name" from MySQL
df_city_db = pd.read_sql("SELECT city_id, city_name FROM city", con=engine)

# 3. We join the airports to their numeric city_ids using the city name.
df_airports_final = pd.merge(df_airports_unique, df_city_db, on="city_name", how="inner")

# We select only the two columns required for the MySQL schema.
df_airports_to_sql = df_airports_final[['arrival_iata', 'city_id']].copy()

# LOADING INTO MYSQL: Insert the data into the 'airports' table
# We use 'replace' because the list of airports for those cities does not change daily
df_airports_to_sql.to_sql(name="airports", con=engine, if_exists="replace", index=False)

DEFINE A FUNCTION TO FIND ALL OF THESE INFORMATION FOR THE DF_AIRPORTS CREATED

In [ ]:
def get_info_flights(df_airports):

    import time
    from datetime import datetime, timedelta
    import pandas as pd
    import requests

    list_flights3 = []
    
    # The headers and official credentials we tested earlier
    headers3 = {
        "Content-Type": "application/json",
        "x-rapidapi-host": 'aerodatabox.p.rapidapi.com',
        "x-rapidapi-key": 'insertyourkeyhere',
    }
    
    # Automatic calculation of tomorrow's date (12-hour range)
    tomorrow_date2 = datetime.now() + timedelta(days=1)
    tomorrow_date3 = tomorrow_date2.strftime("%Y-%m-%d")

    # It loops directly over the table column you pass it.
    for iata_code in df_airports["arrival_iata"].unique():

        url3 = f"https://aerodatabox.p.rapidapi.com/flights/airports/iata/{iata_code}/{tomorrow_date3}T08:00/{tomorrow_date3}T20:00"
        querystring3 = {
            "withLeg": "true",
            "direction": "Arrival",
            "withCancelled": "true",
            "withCodeshared": "true",
            "withCargo": "true",
            "withPrivate": "true",
            "withLocation": "false",
        }

        response3 = requests.get(url3, headers=headers3, params=querystring3)

        if response3.status_code == 200:
            try:
                flights_datas3 = response3.json()
                for flight in flights_datas3.get("arrivals", []):
                    dict_flights3 = {
                        "flight_num": flight.get("number"),
                        "departure_icao": flight.get("departure", {}).get("airport", {}).get("icao", "N/D"),
                        "arrival_time": flight.get("arrival", {}).get("scheduledTime", {}).get("local", "N/D"),
                        "arrival_iata": iata_code,
                    }
                    list_flights3.append(dict_flights3)
                print(f"Voli recuperati per l'aeroporto {iata_code}!")
            except Exception:
                print(f"Errore nella lettura del JSON per l'aeroporto {iata_code}.")
        else:
            print(f"Nessun volo o errore per l'aeroporto {iata_code}: Stato {response3.status_code}")

        time.sleep(3)

    # Final df creation
    df_voli3 = pd.DataFrame(list_flights3)

    if not df_voli3.empty:
        # Perform the merge using the table you passed to the function.
        df_voli_completo3 = pd.merge(df_voli3, df_airports, on="arrival_iata", how="left")
        print("\n Pipeline completata! Tabella creata con successo.")
        return df_voli_completo3
    else:
        print("\nLa lista dei voli è rimasta vuota. Controlla gli stati delle risposte sopra.")
        return df_voli3


In [ ]:
from sqlalchemy import text
import pandas as pd

# Use the function for df_airports_gans
df_risultato_finale = get_info_flights(df_airports_gans)

if not df_risultato_finale.empty:
    # 1. Select only the column requested
    df1 = df_risultato_finale[['flight_num', 'departure_icao', 'arrival_time', 'arrival_iata']].copy()

    # Remove the time zone for compatibility with MySQL DATETIME.
    df1['arrival_time'] = pd.to_datetime(df1['arrival_time']).dt.tz_localize(None)

    # 2. Cleaning for old temporary tables
    with engine.begin() as connection:
        connection.execute(text("DROP TABLE IF EXISTS flight_temporary;"))

    # 3. We load the fresh data into the temporary table
    df1.to_sql(name="flight_temporary", con=engine, if_exists="replace", index=False)

    # 4. Query with temporary disabling of foreign keys to avoid blocking
    upsert_flights_query = """
    INSERT INTO flight (flight_num, departure_icao, arrival_time, arrival_iata)
    SELECT flight_num, departure_icao, arrival_time, arrival_iata 
    FROM flight_temporary
    AS nuovi_voli
    ON DUPLICATE KEY UPDATE 
        departure_icao = nuovi_voli.departure_icao,
        arrival_time = nuovi_voli.arrival_time;
    """

    # We carry out everything within a secure transaction.
    with engine.begin() as connection:
        connection.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
        connection.execute(text(upsert_flights_query))
        connection.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))
        connection.execute(text("DROP TABLE IF EXISTS flight_temporary;"))
